# [Semantic Kernel 04 - Multiple Agents](https://devblogs.microsoft.com/semantic-kernel/introducing-agents-in-semantic-kernel/)

# Constants and Libraries

In [39]:
import os, json, sys, random
from dotenv import load_dotenv # requires python-dotenv
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from semantic_kernel.connectors.ai.open_ai.prompt_execution_settings.azure_chat_prompt_execution_settings import (
    AzureChatPromptExecutionSettings,
)
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior # Auto(), Required() or NoneInvoke()
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.agents import ChatCompletionAgent, AgentGroupChat
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents.chat_message_content import ChatMessageContent

from IPython.display import Markdown, display # nothing to pip install
sys.path.append('./common')
from utils import *  

load_dotenv("./../config/credentials_my.env")

data_path = "./data/data.json"

report_agent_name      = "Report_Agent"
researcher_agent_name  = "Researcher_Agent"
ethicist_agent_name    = "Ethicist_Agent"
economist_agent_name   = "Economist_Agent"
policymaker_agent_name = "PolicyMaker_Agent"

report_service_id      = "Report_Agent"
researcher_service_id  = "Researcher_Agent"
ethicist_service_id    = "Ethicist_Agent"
economist_service_id   = "Economist_Agent"
policymaker_service_id = "PolicyMaker_Agent"

report_system_message      = "You are responsible for creating a report by extracting insights from the chat history."
researcher_system_message  = "You explore and describe the potential capabilities and advancements of the topic."
ethicist_system_message    = "You evaluate the ethical implications of the topic, based on the research findings."
economist_system_message   = "You analyze the economic impact of the topic, based on the ethical evaluations."
policymaker_system_message = "You develop policies to manage the topic, based on the economic analysis."


chatcompletion_service_id   = "chatcompletion_service_id"
instructions                = "you are a clever agent"
content                     = "Toggle the status of my second light."

print(f"os.environ['AZURE_OPENAI_ENDPOINT']: {os.environ['AZURE_OPENAI_ENDPOINT']}")

os.environ['AZURE_OPENAI_ENDPOINT']: https://mmoaiswc-01.openai.azure.com/


# Data

In [36]:
# Open the JSON file and load its content into the 'topics' variable  
with open(data_path, 'r') as file:  
    topics = json.load(file)
    
topic = topic_to_string(random.choice(topics))
print(topic)

Urbanizzazione e sviluppo urbano:
- Crescita delle cittÃ  e pianificazione urbana
- Accesso ai servizi pubblici
- Abitazioni e crisi degli alloggi
- MobilitÃ  sostenibile


# Define the Kernel

In [2]:
kernel = Kernel()
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FC01A7D6A0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Create an AzureChatCompletion AI Service and add it to the Kernel

In [3]:
kernel.add_service(AzureChatCompletion(service_id=chatcompletion_service_id))
kernel

Kernel(retry_mechanism=PassThroughWithoutRetry(), services={'chatcompletion_service_id': AzureChatCompletion(ai_model_id='gpt-4o-for-apim', service_id='chatcompletion_service_id', client=<openai.lib.azure.AsyncAzureOpenAI object at 0x000001FC01A5E2D0>, ai_model_type=<OpenAIModelTypes.CHAT: 'chat'>, prompt_tokens=0, completion_tokens=0, total_tokens=0)}, ai_service_selector=<semantic_kernel.services.ai_service_selector.AIServiceSelector object at 0x000001FC01A7D6A0>, plugins={}, function_invocation_filters=[], prompt_rendering_filters=[], auto_function_invocation_filters=[])

# Enable planning with Function Calling set as Auto()

In [13]:
execution_settings = AzureChatPromptExecutionSettings()
execution_settings.function_choice_behavior= FunctionChoiceBehavior.Auto(auto_invoke=True) # Auto(), Required() or NoneInvoke()
execution_settings

AzureChatPromptExecutionSettings(service_id=None, extension_data={}, function_choice_behavior=FunctionChoiceBehavior(enable_kernel_functions=True, maximum_auto_invoke_attempts=5, filters=None, type_=<FunctionChoiceType.AUTO: 'auto'>), ai_model_id=None, frequency_penalty=None, logit_bias=None, max_tokens=None, number_of_responses=None, presence_penalty=None, seed=None, stop=None, stream=False, temperature=None, top_p=None, user=None, store=None, metadata=None, response_format=None, function_call=None, functions=None, messages=None, function_call_behavior=None, parallel_tool_calls=True, tools=None, tool_choice=None, structured_json_response=False, stream_options=None, extra_body=None)

# Set the logging level for  semantic_kernel.kernel to DEBUG
One of the main benefits of using Semantic Kernel is that it supports enterprise-grade services.<br/>
In this sample, we add the logging service to the kernel to help debug the AI agent.

In [5]:
import logging

logging.basicConfig(
    format="[%(asctime)s - %(name)s:%(lineno)d - %(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S")
    
logging.getLogger("kernel").setLevel(logging.DEBUG)

# Create the agents
The agent `service_id` specified in `ChatCompletionAgent` must match one of the services defined in `Kernel.services`

In [47]:
user_proxy = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="user_proxy",
    description="Orchestrates the multiple personas involved in a discussion",
    kernel=kernel,    
    instructions="""
        Tu sei incaricato di iniziare e moderare la conversazione. 
        Il tuo ruolo è di fornire una piattaforma dove le persone possono interagire, mostrando le loro inclinazioni e abilità.
        Tuo incarico: facilita una conversazione discorsiva e coinvolgente fra Mauro, Aleksa, Gabriel e Federica senza imporre alcun pregiudizio.
    """,    
    execution_settings=execution_settings,
)

In [48]:
mauro = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="mauro",
    description="A Microsoft Cloud Solution Architect expert at Data Engineering",
    kernel=kernel,    
    instructions="""
        Tu sei Mauro. Mauro è un Cloud Solution Architect specializzato in Data Engineering, 
        esperto nella progettazione e implementazione di soluzioni per la gestione e l'analisi di grandi volumi di dati, 
        con un forte impegno verso l'ottimizzazione delle prestazioni e l'integrità dei dati.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """,    
    execution_settings=execution_settings,
)

In [49]:
aleksa = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="aleksa",
    description="A Microsoft Cloud Solution Architect expert at Data Science and Generative AI",
    kernel=kernel,    
    instructions="""
        Tu sei Aleksa. Sei un Cloud Solution Architect specializzata in Data Science, con una profonda conoscenza 
        delle tecniche di machine learning, intelligenza artificiale e generative AI, appassionata di trasformare dati complessi 
        in insights utili per guidare le decisioni aziendali.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """,    
    execution_settings=execution_settings,
)

In [50]:
gabriel = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="gabriel",
    description="A Microsoft Cloud Solution Architect expert at Security",
    kernel=kernel,    
    instructions="""
        Tu sei Gabriel. Sei Cloud Solution Architect esperto di sicurezza, dedicato a progettare e implementare 
        soluzioni di sicurezza cloud robuste e scalabili, con un forte impegno verso la protezione dei dati e la conformità alle normative.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """,    
    execution_settings=execution_settings,
)

In [51]:
federica = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name="federica",
    description="A Microsoft Cloud Solution Architect expert in Development Tools",
    kernel=kernel,    
    instructions="""
        Tu sei Federica. Sei Cloud Solution Architect specializzata in tecnologie di sviluppo, con una vasta esperienza 
        nella creazione di applicazioni cloud-native e microservizi, sempre alla ricerca di innovazioni che migliorino 
        l'efficienza e la scalabilità delle soluzioni software.
        I tuoi messaggi sono brevi. Ciascuno messaggio termina lasciando la parola ad un altro interlocutore.
    """,    
    execution_settings=execution_settings,
)

In [38]:
report_agent = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name=report_agent_name,
    description=report_system_message,
    kernel=kernel,
    instructions=report_system_message,
    execution_settings=execution_settings,
)

researcher_agent = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name=researcher_agent_name,
    kernel=kernel,
    instructions=researcher_system_message,
    execution_settings=execution_settings,
)

ethicist_agent = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name=ethicist_agent_name,
    kernel=kernel,
    instructions=ethicist_system_message,
    execution_settings=execution_settings,
)

economist_agent = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name=economist_agent_name,
    kernel=kernel,
    instructions=economist_system_message,
    execution_settings=execution_settings,
)

policymaker_agent = ChatCompletionAgent(
    service_id=chatcompletion_service_id,
    name=policymaker_agent_name,
    kernel=kernel,
    instructions=policymaker_system_message,
    execution_settings=execution_settings,
)

# Create a user message and add it to a blank history

In [37]:
history = ChatHistory()

# Add the user message
history.add_message(ChatMessageContent(role=AuthorRole.USER, content=topic))

history

ChatHistory(messages=[ChatMessageContent(inner_content=None, ai_model_id=None, metadata={}, content_type='message', role=<AuthorRole.USER: 'user'>, name=None, items=[TextContent(inner_content=None, ai_model_id=None, metadata={}, content_type='text', text='Urbanizzazione e sviluppo urbano:\n- Crescita delle cittÃ\xa0 e pianificazione urbana\n- Accesso ai servizi pubblici\n- Abitazioni e crisi degli alloggi\n- MobilitÃ\xa0 sostenibile', encoding=None)], encoding=None, finish_reason=None)])

# Create Group Chat

In [41]:
chat = AgentGroupChat(
    agents = [user_proxy, mauro, aleksa, gabriel, federica]
)
chat.add_chat_message(history)      

<coroutine object AgentChat.add_chat_message at 0x000001FC2E2BCEE0>

In [44]:
async for response in chat.invoke():
    print(response)

Sure, I'd be happy to help with that. Please provide the chat history that you need insights from.


In [45]:
response

ChatMessageContent(inner_content=ChatCompletion(id='chatcmpl-AnX6ASgwZt40HQ9lcZPDIv9iQj7dQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Sure, I'd be happy to help with that. Please provide the chat history that you need insights from.", refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1736368666, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_04751d0b65', usage=CompletionUsage(completion_tokens=21, prompt_tokens=26, total_tokens=47, completion_tokens_details=None, prompt_tokens_details=None), prompt_filter_results=[{'prompt_index': 0, 'content_filter_results': {}}]), ai_model_id='gpt-4o-for-apim', metadata={'log

# Generate the agent response(s)

In [ ]:
async for response in agent.invoke(history):
  print(response)

# Additional tests. Run multiple times to toggle the first light.

In [ ]:
history = ChatHistory() # initially blank
history.add_user_message("Toggle the first light and give me the status of all my lights.")

async for response in agent.invoke(history):
  print(response)

In [ ]:
history

In [ ]:
for cmc in history.messages: # ChatMessageContent
    if not cmc.inner_content is None:
        for choice in cmc.inner_content.choices:
            for tc in choice.message.tool_calls:
                print (f"Call {tc.function.name}({tc.function.arguments})")